#  **ICT303 - Assignment 2**

**Your name:**

**Student ID: **

**Email: **

In this assignment, you will build and train a deep learning model for solving a problem of your choice.


You are required to:
- Think of a practical problem that you would like to solve. The problem can be related, but not limted to, object detection and recognition from images, text analysis, speech analysis, image unpainting, converting images to artistic painting, action recognition (from images or videos), image to text (i.e., generating textual description for images or videos), or texrt to image (generating images from text) etc.,
- Find an appropriate data set to train and test the model you will develop. Note that the dataset should contain enough data (with groundtruth labels) so that when used for training, the model can generalize well to unseen data.
- Design a neural network that will solve the problem
- Train the neural network on your training data and then evaluate its performance on test data
- Analyze the performance of the network you developed and discuss its limitations.

**What to submit:**
- A colab notebook that describes:
 - The problem you would like to solve **[10 Marks]**
 - The dataset that you will use to train and test the deep learning model that you will develop **[10 marks]**
 - A diagram that describes the architecture of the neural network that you developed **[10 marks]**
 - Performance curves - this is includes the loss curves as well as the accuracy **[10 marks]**
 - A discussion, analysis and justification of the different choices you made and their effect on the performance **[15 marks]**
 - A discussion, analysis of the limitations of your method. You can also show failure cases and try to understand why did it fail on these cases **[15 marks]**

- Source code that runs - this includes both code for training and testing **[30 marks]**

You also need to submit the dataset you used for training and testing, or alternatively provide the code that downloads the data.

Make sure you reference all sources from which you took information.

You are allowed to use existing neural networks (not required to implement them from scratch). But, you must customize the architecture to the problem you want to solve.

**Where to find datasets?**
- [Kaggle competition](https://www.kaggle.com/c/dog-breed-identification) is a good source.
- You can also look at this [wikipedia site[(https://en.wikipedia.org/wiki/List_of_datasets_for_machine-learning_research)

If you are thinking of a specific problem and were unable to find a suitable dataset, please talk to me during the lecture or lab and we will search together.

**Recommended timeline**
- Week 1 of Assignement release: identify 2 or 3 problems of interest, a find dataset for each of the problem and try to understand how to load the data and how it is organised. Discuss it with the UC during the lab session or via email.
- Week 2:
 - Make sure your dataloader works proply and you are able to load the data and structure it in a way that neural networks can use them.
 - Design your network architecture
- Week 3: Network architecture implemented and training and testing done. Evaluate the performance
- Week 4: Finetune the network architecture and the hyperparameters to improve the performance. Write the report for submission.

It is highly recommended that you follow this timeline. The earlier you start training and testing, the more time you will have to finetune your solution and achieve a better performance.


# Movie Recommendation using Neural Collaboration Filtering Model

dataset - https://www.kaggle.com/datasets/samlearner/letterboxd-movie-ratings-data?

In [15]:
print("tensorboard extension loaded")

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
tensorboard extension loaded


## import and dependencies

In [16]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import tqdm
import os
import shutil
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import mean_squared_error
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import kagglehub

# Download latest version
path = kagglehub.dataset_download("samlearner/letterboxd-movie-ratings-data")

print("Path to dataset files:", path)


# Load CSV files



Path to dataset files: /home/kim/.cache/kagglehub/datasets/samlearner/letterboxd-movie-ratings-data/versions/6


In [17]:
filtered = merged_data_df[merged_data_df['movie_title'].str.contains('lord of the rings', case=False, na=False)]
print(filtered[['movie_title', 'popularity']])

                                                movie_title  popularity
1021          The Lord of the Rings: The Return of the King      79.157
1131                  The Lord of the Rings: The Two Towers      74.387
1452      The Lord of the Rings: The Fellowship of the Ring      90.731
4390                  The Lord of the Rings: The Two Towers      74.387
4416      The Lord of the Rings: The Fellowship of the Ring      90.731
...                                                     ...         ...
11074984  The Lord of the Rings: The Fellowship of the Ring      90.731
11078776      The Lord of the Rings: The Return of the King      79.157
11078777              The Lord of the Rings: The Two Towers      74.387
11078778  The Lord of the Rings: The Fellowship of the Ring      90.731
11078865  The Lord of the Rings: The Fellowship of the Ring      90.731

[12450 rows x 2 columns]


In [14]:
print(merged_data_df)

                             _id_x             movie_id  rating_val  \
0         5fc57c5d6758f6963451a07f           feast-2014           7   
1         5fc57c5d6758f6963451a063          loving-2016           7   
2         5fc57c5d6758f6963451a0ef     scripted-content           7   
3         5fc57c5d6758f6963451a060           the-future           4   
4         5fc57c5c6758f69634519398                 mank           5   
...                            ...                  ...         ...   
11079661  6239f4f1a936b95600b3d798              alien-3           6   
11079662  6239f4f1a936b95600b3d799  battleship-potemkin           7   
11079663  6239f4f1a936b95600b3d79e               pusher           6   
11079664  6239f4f1a936b95600b3d7a1    wild-strawberries           7   
11079665  6239f4f1a936b95600b3d7a2               x-2022           5   

             user_id                     _id_y  \
0         deathproof  5fc880726758f69634df0bca   
1         deathproof  5fc879b26758f69634bf9665 

## Neural Collboration Filtering Model Architecture



## NCF Model Class

In [18]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=8, lr=1e-3, optimizer_type="adam"):
        super(NCF, self).__init__()
        self.lr = lr
        self.optimizer_type = optimizer_type

        # Embeddings for GMF
        self.user_embedding_gmf = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_gmf = nn.Embedding(num_items, embedding_dim)

        # Embeddings for MLP
        self.user_embedding_mlp = nn.Embedding(num_users, embedding_dim)
        self.item_embedding_mlp = nn.Embedding(num_items, embedding_dim)

        # MLP Layers
        self.mlp_layers = nn.Sequential(
            nn.Linear(embedding_dim * 2, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU()
        )

        # Final prediction layer
        self.output_layer = nn.Linear(embedding_dim + 16, 1)  # GMF (8) + MLP output (16)

    def forward(self, user_indices, item_indices):
        # GMF embedded
        gmf_user = self.user_embedding_gmf(user_indices)
        gmf_item = self.item_embedding_gmf(item_indices)
        # GMF output
        gmf_output = gmf_user * gmf_item  # element-wise product

        # MLP embedded
        mlp_user = self.user_embedding_mlp(user_indices)
        mlp_item = self.item_embedding_mlp(item_indices)

        # MLP input
        mlp_input = torch.cat([mlp_user, mlp_item], dim=-1)

        #MLP output
        mlp_output = self.mlp_layers(mlp_input)

        # NeuMF Layer (GMF x MLP)
        combined = torch.cat([gmf_output, mlp_output], dim=-1)

        # Final prediction
        prediction = self.output_layer(combined).squeeze(-1)

        return prediction

    def loss(self, y_hat, y):
        return nn.MSELoss()(y_hat, y)

    def configure_optimizers(self):
        if self.optimizer_type == "adam":
            return optim.Adam(self.parameters(), lr=self.lr)
        elif self.optimizer_type == "sgd":
            return optim.SGD(self.parameters(), lr=self.lr)
        else:
            return optim.Adam(self.parameters(), lr=self.lr)

## Trainer Class

In [19]:
class Trainer:

  def __init__(self, tb, n_epochs = 3):
    self.max_epochs = n_epochs
    self.writer = tb  # the tensorboard instance
    return

  def fit(self, model, data, validation_data):
    self.data = data
    self.validation_data = validation_data

    # configure the optimizer
    self.optimizer = model.configure_optimizers()
    #self.scheduler = StepLR(self.optimizer, step_size=5, gamma=0.1)
    self.model     = model

    for epoch in range(self.max_epochs):
      print(f"\nEpoch {epoch + 1}/{self.max_epochs}")
      self.fit_epoch()
      self.validate_epoch()
      #self.scheduler.step()
      # Logging the average training loss so that it can be visualized in the tensorboard
      self.writer.add_scalar("Training Loss", self.avg_training_loss, epoch)
      self.writer.add_scalar("Validation Loss", self.avg_val_loss, epoch)

    print("Training process has finished")

  def fit_epoch(self):

    self.model.train()
    current_loss = 0.0
    self.avg_training_loss = 0.0

    # iterate over the DataLoader for training data
    for i, data in enumerate(tqdm(self.data, desc="Training")):
      # Get input
      (inputs, target) = data
      user, item = inputs  # instead of: inputs, target
      user, item, target = user.to(device), item.to(device), target.to(device)


      # Clear gradient buffers because we don't want any gradient from previous
      # epoch to carry forward, dont want to cummulate gradients
      self.optimizer.zero_grad()

      # get output from the model, given the inputs
      outputs = self.model(user, item)

      # get loss for the predicted output
      loss = self.model.loss(outputs, target)

      # get gradients w.r.t to the parameters of the model
      loss.backward()

      # update the parameters (perform optimization)
      self.optimizer.step()

      # Let's print some statistics (average of the training loss over minibatches of 500 data items)
      current_loss += loss.item()

      # Adding training loss
      self.avg_training_loss += loss.item()

      if i % 500 == 499:
          print('Loss after mini-batch %5d: %.3f' %
                (i + 1, current_loss / 500))
          current_loss = 0.0

    # The average training loss
    self.avg_training_loss = self.avg_training_loss / i # to get the average
    print(f"Training Loss (avg): {self.avg_training_loss:.4f}")

  def validate_epoch(self):

    self.model.eval()
    total_loss = 0.0
    self.avg_val_loss = 0.0

    with torch.no_grad():
      # iterate over the DataLoader for training data
      for i, data in enumerate(self.validation_data):
        # Get input
        (inputs, target) = data
        user, item = inputs  # instead of: inputs, target
        user, item, target = user.to(device), item.to(device), target.to(device)

        # get output from the model, given the inputs
        outputs = self.model(user, item)

        # get loss for the predicted output
        loss = self.model.loss(outputs, target)

        total_loss += loss.item()

      # The average training loss
      self.avg_val_loss = total_loss / (i + 1) # to get the   average
      print(f"Validation Loss (avg): {self.avg_val_loss:.4f}")

In [20]:
class Dataloader:
    def __init__(self, dataframe, batch_size=1024, validation_split=0.2):
        self.data = dataframe.copy()
        self.data["label"] = self.data["rating_val"] / 2.0  # Normalized to 0–2.5

        # Map user and item IDs to indices
        self.user2idx = {u: i for i, u in enumerate(self.data["user_id"].unique())}
        self.item2idx = {m: i for i, m in enumerate(self.data["movie_id"].unique())}

        self.data["user"] = self.data["user_id"].map(self.user2idx)
        self.data["item"] = self.data["movie_id"].map(self.item2idx)

        self.num_users = len(self.user2idx)
        self.num_items = len(self.item2idx)
        self.data = self.data[["user", "item", "label"]]

        from sklearn.model_selection import train_test_split
        self.train, self.val = train_test_split(self.data, test_size=validation_split, random_state=42)

    def get_dataloaders(self):
        return self.train, self.val



class NCFDataset(Dataset):
    def __init__(self, data):
        self.data = data.reset_index(drop=True)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        user = torch.tensor(row["user"], dtype=torch.long)
        item = torch.tensor(row["item"], dtype=torch.long)
        label = torch.tensor(row["label"], dtype=torch.float)
        return (user, item), label


In [21]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

#Hyperparameters
embedding_dim=32
lr=0.001
batch_size=8192
n_epochs=10


Using device: cuda


In [23]:
movie_df = pd.read_csv(os.path.join(path,'movie_data.csv'), engine='python')
ratings_df = pd.read_csv(os.path.join(path,'ratings_export.csv'), engine='python')
merged_data_df = pd.merge(ratings_df, movie_df, on='movie_id')
filtered_reviews = merged_data_df[merged_data_df['popularity'] > 50]


In [24]:
dl = Dataloader(filtered_reviews, batch_size=batch_size, validation_split=0.2)

train_dataset,val_dataset = dl.get_dataloaders()

train_dataset = NCFDataset(train_dataset)
val_dataset = NCFDataset(val_dataset)


train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size,
                          num_workers=4, pin_memory=True)

print ("Done")

Done


In [ ]:
model = NCF(num_users=dl.num_users, num_items=dl.num_items, embedding_dim=embedding_dim, lr=lr)
model.to(device)


if os.path.exists('runs/ncf_model'):
  shutil.rmtree('runs/ncf_model')
writer = SummaryWriter(log_dir="runs/ncf_model")

trainer = Trainer(tb=writer, n_epochs=n_epochs)
trainer.fit(model, train_loader, val_loader)





Epoch 1/10


Training:  87%|█████████████████████████████████████████████████████████████▊         | 108/124 [00:20<00:02,  5.71it/s]

In [ ]:
import numpy as np
def compute_rmse(model, dataloader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for (inputs, labels) in dataloader:
            user, item = inputs
            user, item = user.to(device), item.to(device)
            labels = labels.to(device)

            preds = model(user, item)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    mse = mean_squared_error(all_labels, all_preds)
    rmse = np.sqrt(mse)
    return rmse



rmse = compute_rmse(model, val_loader)
print(f"Test RMSE: {rmse:.4f}")



Test RMSE: 0.7523


In [ ]:
def recommend_top_k(model, dl: Dataloader, user_name: str, k=10):
    import pandas as pd
    import os
    import torch

    model.eval()

    if user_name not in dl.user2idx:
        print(f"User '{user_name}' not found in dataset.")
        return

    movie_data_path = os.path.join(path, "movie_data.csv")

    # Get user's top-rated movies
    df = pd.read_csv(dl.file_path)
    user_df = df[df["user_id"] == user_name]
    user_df = user_df.sort_values(by="rating_val", ascending=False).head(20)
    user_idx = dl.user2idx[user_name]

    try:
        movie_df = pd.read_csv(movie_data_path, usecols=["movie_id", "movie_title"],
                               engine='python', on_bad_lines='skip', encoding='utf-8', sep=',')
    except Exception as e:
        print(f"Error reading movie data: {e}")
        return

    merged_df = pd.merge(user_df, movie_df, on="movie_id", how="left")
    print(f"\n🎬 Top 20 Movies Rated by '{user_name}':")
    for idx, row in enumerate(merged_df.itertuples(), 1):
         print(f"{idx}. {row.movie_title} — Rating: {row.rating_val}")

    # Find movies not rated by the user
    seen_movie_ids = set(user_df["movie_id"])
    unseen_movies = [(i, m) for m, i in dl.item2idx.items() if m not in seen_movie_ids]

    if not unseen_movies:
        print("No unseen movies to recommend.")
        return

    # Predict ratings
    user_tensor = torch.tensor([user_idx] * len(unseen_movies), dtype=torch.long).to(device)
    item_tensor = torch.tensor([idx for idx, _ in unseen_movies], dtype=torch.long).to(device)

    with torch.no_grad():
        preds = model(user_tensor, item_tensor)

    # Top-k predictions
    top_indices = preds.cpu().numpy().argsort()[::-1][:k]
    top_movies = [unseen_movies[i][1] for i in top_indices]
    movie_dict = dict(zip(movie_df["movie_id"], movie_df["movie_title"]))

    print(f"\n🔮 Top {k} Recommendations for '{user_name}':")
    for rank, movie_id in enumerate(top_movies, start=1):
        title = movie_dict.get(movie_id, "Unknown Title")
        print(f"{rank}. {title}")

In [ ]:
recommend_top_k(model, dl, user_name="deathproof", k=20)


🎬 Top 20 Movies Rated by 'deathproof':
1. Incendies — Rating: 10
2. The Social Network — Rating: 10
3. The Fall — Rating: 10
4. Beetlejuice — Rating: 10
5. Little Miss Sunshine — Rating: 10
6. Death Proof — Rating: 10
7. Garden State — Rating: 10
8. The Lord of the Rings: The Return of the King — Rating: 10
9. Drop Dead Gorgeous — Rating: 10
10. Now and Then — Rating: 10
11. Clueless — Rating: 10
12. Kill Bill: The Whole Bloody Affair — Rating: 10
13. World of Tomorrow Episode Two: The Burden of Other People's Thoughts — Rating: 10
14. Don't Breathe — Rating: 10
15. The Bling Ring — Rating: 10
16. Mad Max: Fury Road — Rating: 10
17. Cameraperson — Rating: 10
18. The Room — Rating: 10
19. World of Tomorrow — Rating: 10
20. Hocus Pocus — Rating: 10

🔮 Top 20 Recommendations for 'deathproof':
1. Old Digs
2. Juliane Lorenz über Lili Marleen
3. Objetos sexuales
4. Cuando el diablo sopla
5. nan
6. Jugandose la vida
7. Skyfall
8. Heval
9. Numéro zéro
10. The Polarman
11. nan
12. The Peach Tr

In [ ]:
tensorboard --logdir runs

In [ ]:
#TODO: Rename num_user/num_item